# AI Productivity Maximizer for Students (Runnable Version)

This notebook is a simplified, stable version (similar to your Phase 2 style).

Dataset: **student_habits_performance.csv**

What it does:
1. Loads and cleans data
2. Builds performance classification model (High vs Low)
3. Evaluates model
4. Generates a simple priority-based study schedule

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

## 2. Load Dataset

In [2]:
# IMPORTANT: Keep the CSV in same folder as notebook
df = pd.read_csv('student_habits_performance.csv')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'student_habits_performance.csv'

## 3. Dataset Overview

In [ ]:
print('Shape:', df.shape)
df.info()

## 4. Handle Missing Values + Basic Cleaning

In [ ]:
print('Missing values before:')
print(df.isnull().sum())

# Remove ID column if exists
if 'student_id' in df.columns:
    df.drop(columns=['student_id'], inplace=True)

# Fill missing categorical values with mode
df.fillna(df.mode().iloc[0], inplace=True)

print('\nMissing values after:')
print(df.isnull().sum())

## 5. Encode Categorical Features

In [ ]:
le_dict = {}
for col in df.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le

df.head()

## 6. Create Target Variable (Binary Performance)

In [ ]:
# exam_score >= 70 => High Performance (1), else Low Performance (0)
df['performance'] = df['exam_score'].apply(lambda x: 1 if x >= 70 else 0)
df.drop(columns=['exam_score'], inplace=True)

df['performance'].value_counts()

## 7. EDA

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

## 8. Train/Test Split

In [ ]:
X = df.drop(columns=['performance'])
y = df['performance']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Train:', X_train.shape)
print('Test :', X_test.shape)

## 9. Train Model

In [ ]:
model = RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42)
model.fit(X_train, y_train)

## 10. Evaluate Model

In [ ]:
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print('Accuracy :', round(acc,4))
print('Precision:', round(prec,4))
print('Recall   :', round(rec,4))
print('F1 Score :', round(f1,4))

print('\nClassification Report:\n')
print(classification_report(y_test, y_pred))

## 11. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## 12. Simple AI Study Scheduler (Rule-Based Prototype)

In [ ]:
from datetime import datetime

tasks = [
    {'task':'AI Assignment', 'deadline':'2026-03-20', 'difficulty':5, 'hours':4},
    {'task':'DB Quiz Prep', 'deadline':'2026-03-17', 'difficulty':3, 'hours':2},
    {'task':'OS Midterm Prep', 'deadline':'2026-03-22', 'difficulty':4, 'hours':5},
    {'task':'Math Final Prep', 'deadline':'2026-03-28', 'difficulty':5, 'hours':6}
]

today = datetime.strptime('2026-03-15', '%Y-%m-%d')

def priority_score(task):
    dline = datetime.strptime(task['deadline'], '%Y-%m-%d')
    days_left = max((dline - today).days, 1)
    urgency = 1 / days_left
    return (0.5*urgency) + (0.3*(task['difficulty']/5)) + (0.2*(task['hours']/6))

for t in tasks:
    t['priority'] = round(priority_score(t),4)

tasks_sorted = sorted(tasks, key=lambda x: x['priority'], reverse=True)
schedule_df = pd.DataFrame(tasks_sorted)
schedule_df

## 13. Save Outputs

In [ ]:
import os, joblib

os.makedirs('artifacts', exist_ok=True)
joblib.dump(model, 'artifacts/performance_model.pkl')
schedule_df.to_csv('artifacts/study_schedule.csv', index=False)

print('Saved: artifacts/performance_model.pkl')
print('Saved: artifacts/study_schedule.csv')